In [1]:
import torch
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np, torch, scipy.sparse as sp
import torch
import scvi

import sys 
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))
from model.lr_dec import *

In [2]:
base_dir = 'data/GP1/processed'

adata = sc.read_h5ad(f'{base_dir}/GP1.h5ad')

adata.obs_names_make_unique()

In [7]:
adata.obs["cluster"] = adata.obs["cluster"].astype("category")
adata.obs["cluster"] = adata.obs["cluster"].cat.remove_unused_categories()

print(adata.obs["cluster"].unique())

lr_pairs = pd.read_csv("data/lrpairs/human/LR_pairs.csv")
print(lr_pairs.iloc[:1, 2:4])
ligand_list = lr_pairs['ligand'].unique()
receptor_list = lr_pairs['receptor'].unique()

gene_idx = pd.Index(adata.var_names.astype(str))
ligand   = gene_idx.intersection(pd.Index(ligand_list))
receptor = gene_idx.intersection(pd.Index(receptor_list))
ligand_df   = pd.DataFrame(ligand.sort_values(), columns=["ligand"])
receptor_df = pd.DataFrame(receptor.sort_values(), columns=["receptor"])
print("ligand gene num: ", len(ligand))
print("receptor gene num: ", len(receptor))
feat = adata.var_names.astype(str).values

X = adata.layers['counts']
X = X.toarray() if sp.issparse(X) else X

expr = pd.DataFrame(X, index=adata.obs_names, columns=feat)

expr_L = expr.loc[:, expr.columns.intersection(ligand)]
expr_R = expr.loc[:, expr.columns.intersection(receptor)]

print(expr_L.iloc[:2, :2])
print(expr_R.iloc[:2, :2])

['8', '7', '12', '6', '1', '9']
Categories (6, object): ['1', '6', '7', '8', '9', '12']
  ligand   receptor
0  TGFB1  TGFbR1_R2
ligand gene num:  507
receptor gene num:  347
                           AGRN  UTS2
AAAGGGCAGCTTGAAT-1-normal   3.0   0.0
AAATGCTCGTTACGTT-1-normal   0.0   0.0
                           TNFRSF18  TNFRSF4
AAAGGGCAGCTTGAAT-1-normal       0.0      0.0
AAATGCTCGTTACGTT-1-normal       1.0      0.0


In [9]:
model_dir = "outputs/GP1/ckpt/scvi/scanvi"
model = scvi.model.SCANVI.load(model_dir)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File /home/zhouyj/project/src/GP1_ckpt/scvi/scanvi/model.pt already downloaded                            


## TRAIN DECORDER

In [12]:
OUT_DIR = "outputs/GP1/ckpt/exp_decoder"
SAMPLE_IDS = ["normal", "cancer"]

genes = sorted(set(map(str, ligand)) | set(map(str, receptor)))

results = train_count_decoder_pairs(
    adata=adata,
    model=model,
    genes=genes,
    sample_ids=SAMPLE_IDS,
    sample_key="status",
    out_dir=OUT_DIR,
    sample_prefix="decoder_counts_pair",
    layer_counts="counts",
    skip_if_exists=False,
    seed=2025,
    test_size=0.2,
    batch_size=512,
    epochs=1000,
    patience=50,
    lr=1e-3,
    device="cuda:0",
)
for r in results:
    print(r)

INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


normal_cancer ep 416/1000:  42%|▊ | 415/1000 [00:05<00:07, 76.50it/s, tr=2.2027 va=1.6631]

DecoderTrainResult(pair_id='normal_cancer', n_cells=391, n_genes=808, best_val=1.6631022155815265, decoder_ckpt_path='/home/zhouyj/project/src/GP1_ckpt/exp_decoder/decoder_counts_pair_normal_cancer.pt', decoder_stat_path='/home/zhouyj/project/src/GP1_ckpt/exp_decoder/decoder_counts_pair_normal_cancer.json')
